# Professional Digital Twin — Project Development Notebook

This notebook documents the development of my first independent Agentic AI application: a professional Digital Twin.

Unlike the Foundations notebook, which records how I learned the underlying concepts, this notebook focuses on the **actual project**: its architecture, implementation, tools, Agent Loop, interface, deployment, testing, and ongoing improvements.

**Live Demo:** https://huggingface.co/spaces/Sollie/twin

**Status:** Active Development

## 1. Project Overview

The Digital Twin is a conversational AI application that represents my professional background, skills, experience, and career journey. It is designed for visitors such as potential clients, collaborators, or future employers.

The current implementation combines:
- professional context from `linkedin.pdf` and `summary.txt`
- an OpenAI-powered conversational model
- conversation history
- function/tool calling
- a no-framework Agent Loop
- Pushover notifications
- a Gradio interface
- modular Python files
- Hugging Face deployment

## 2. Project Structure

```text
professional-digital-twin/
├── README.md
├── professional-digital-twin.ipynb
├── app.py
├── context.py
├── tools.py
├── styles.py
├── requirements.txt
├── linkedin.pdf
└── summary.txt
```

The notebook is a development record. The Python modules are the application itself.

## 3. Application Setup

The application uses Python to connect the user interface, professional context, OpenAI model, and external tools. Sensitive credentials are loaded from environment variables rather than being written into the source code.

In [1]:
# 18. Expand the Digital Twin with practical tools
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr


c:\Users\USER PC\projects\Agentic-AI-Engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

load_dotenv(override=True)
openai = OpenAI()


## 4. Professional Context

The Digital Twin needs reliable information about the person it represents. The current project uses two source files:

- `linkedin.pdf` — professional profile and experience
- `summary.txt` — additional personal/professional context

This is the project's context-engineering layer.

In [6]:
# 27. Reload the Digital Twin's source context
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()


## 5. Digital Twin Behavior

The system prompt defines the Digital Twin's identity, professional scope, behavior, and rules for handling unknown information.

The current design instructs the agent to stay focused on career, background, skills, and experience; avoid inventing information; record unanswered questions; and capture contact interest when appropriate.

In [7]:
system_prompt = f"""
# Your role

You are Solomon Olusanya's professional Digital Twin.

You are an AI representation of Solomon, designed to speak with visitors
who may be potential employers, clients, collaborators, recruiters, or
people interested in his work.

You should represent Solomon accurately and naturally.

You are not Solomon himself. If someone asks whether you are AI, explain
clearly that you are an AI Digital Twin representing him.

# About Solomon

{summary}

# Professional context

Here is Solomon's professional background and experience:

{linkedin}

Use this information as your primary source of truth when answering
questions about Solomon's career, education, skills, experience, projects,
and professional interests.

# Personality and communication style

Speak naturally and conversationally while remaining professional.

Solomon tends to communicate in a straightforward and thoughtful way.
He prefers clear explanations and practical answers rather than overly
formal or unnecessarily complicated language.

He is curious and often asks "why?" because he likes understanding how
things work rather than simply memorising information.

His communication can include light humour or a casual expression when
appropriate, but professional conversations should remain professional.

Do not make every response sound corporate or robotic.

Do not exaggerate his personality, experience, achievements, or expertise.

# Professional boundaries

Prioritise questions about:

- Solomon's career
- Education and academic background
- Technical skills
- Data analytics
- AI and Agentic AI
- Agriculture and technology
- Projects
- Professional experience
- Career interests
- Ways to work with or contact him

If a question is unrelated to Solomon's professional background, politely
redirect the conversation towards a relevant professional topic.

Do not discuss private or sensitive personal matters.

# Accuracy

Never invent information.

If the answer is available in the provided context, answer confidently.

If you are unsure or the information is not available in the context,
use the appropriate tool to record the unanswered question and tell the
visitor honestly that you don't have that information.

Do not guess simply to keep the conversation going.

# Contact requests

If a visitor expresses interest in contacting, hiring, collaborating with,
or working with Solomon:

1. Ask for their email address if they have not provided it.
2. Use the appropriate tool to record their contact information.
3. Confirm that their information has been recorded.

# Unanswered questions

If a visitor asks a professional question that you cannot answer from the
available context:

1. Use the appropriate tool to record the question.
2. Tell the visitor that you don't currently have enough information to
   answer accurately.
3. Do not fabricate an answer.

# Overall behaviour

Be helpful, honest, conversational, and professional.

Represent Solomon's actual experience rather than trying to make him
sound more impressive than he is.

The goal is not simply to answer questions.

The goal is to give visitors a useful and authentic representation of
Solomon's professional background and current direction in Agentic AI.
"""

## 6. Tooling and External Actions

The Digital Twin can move beyond generating text by calling Python tools. The current project includes Pushover-based external notification functionality and tools for recording contact interest and unanswered questions.

In [8]:
# 19. Load Pushover configuration

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")


Pushover user found and looks good
Pushover token found and looks good


In [9]:
# 20. Create a notification function
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)


In [10]:
# 21. Tool for recording contact interest
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"


In [11]:
# 22. Tool for unanswered questions
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return "OK"


### Tool schemas

The Python functions define what the tools actually do. The JSON schemas describe those tools to the model so it can decide when to request them.

In [12]:
# 23. Describe the contact tool to the LLM
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The user's name, if they provided it"},
            "notes": {"type": "string", "description": "Any additional info about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [13]:
# Describe the unknown-question tool
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
        },
        "required": ["question"],
        "additionalProperties": False
    }
}


In [14]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]


## 7. Dynamic Tool Execution

The first implementation used explicit `if/elif` mapping between tool names and Python functions. The project then moved to dynamic lookup with `globals()`.

The resulting flow is:

**LLM chooses a tool → Python finds the function → JSON arguments become Python arguments → tool executes → result returns to the LLM**

In [15]:
# 24. First approach: manually map tool names to functions

def handle_tool_calls_with_manual_if(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        # THE BIG IF STATEMENT!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results


In [16]:
# 25. Understand `globals()`
globals()["record_unknown_question"]("this is a really hard question")


Push: Recording this is a really hard question asked that I couldn't answer


'OK'

In [17]:
# 26. Dynamically execute the requested tool

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "No tool found"
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results


## 8. Agent Loop

The Digital Twin becomes an agent when it can request a tool, receive the tool result, and continue the conversation rather than stopping after the first model response.

```text
User message
     ↓
OpenAI model
     ↓
Tool call? ── No ──→ Final response
     │
    Yes
     ↓
Python tool
     ↓
Tool result
     ↓
OpenAI model
     ↓
Final response
```

In [18]:
# 29. Build the complete Lab 4 Agent Loop
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    return response.choices[0].message.content


## 9. Gradio Interface

Gradio provides the public conversational interface around the `chat()` function.

In [19]:
# 30. Put the complete Digital Twin behind Gradio
gr.ChatInterface(chat).launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 10. From Notebook to Application

The project was separated into reusable Python modules so the Digital Twin could run independently of Jupyter.

- `app.py` — application and chat interface
- `context.py` — professional/personal context
- `tools.py` — tool functions
- `styles.py` — interface styling
- `linkedin.pdf` and `summary.txt` — source context
- `requirements.txt` — dependencies

The local application can be started with:

```bash
uv run app.py
```

## 11. Deployment

The application is publicly deployed as a Gradio Space on Hugging Face.

**Live Demo:** https://huggingface.co/spaces/Sollie/twin

Deployment secrets are kept outside the repository:

```text
OPENAI_API_KEY
PUSHOVER_USER
PUSHOVER_TOKEN
```

The application code and deployment configuration are kept separate from the secret values.

## 12. Testing and Current Issues

The deployed application is currently treated as an active development project rather than a finished production system.

During testing, I identified an issue with the intended out-of-scope/notification behavior: a question outside the Digital Twin's professional scope did not produce the expected Pushover notification. This is now a debugging task rather than something to hide from the project record.

Other planned improvements include:
- improving tool reliability
- strengthening scope control
- adding a work-focused evaluator
- improving Agent Loop behavior
- expanding useful tools
- improving context and retrieval
- exploring memory and higher-level agent frameworks

## 13. What This Project Represents

This project is the transition from learning individual Agentic AI concepts to combining them into a deployed application.

The development path is:

**Concepts → implementation → tools → Agent Loop → modular application → deployment → testing → iteration**

The Foundations notebook documents how I learned these mechanics. This notebook documents how those mechanics became an actual project.